In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import random
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import seaborn as sns
import re

In [ ]:
def parse_ids_to_dict(text_content):
    # Initialize the dictionary to store the IDs
    cardiac_ids = {}
    
    # Use a pattern to find all disease categories in the text
    disease_pattern = r"([A-Za-z\s]+) IDs: (.*?)(?:\nNumber|\Z)"
    
    # Find all matches in the text
    disease_matches = re.finditer(disease_pattern, text_content, re.DOTALL)
    
    # Process each disease category found
    for match in disease_matches:
        disease_name = match.group(1).strip()
        ids_str = match.group(2).strip()
        
        # Convert string of IDs to list of integers
        try:
            ids_list = [int(id_str.strip()) for id_str in ids_str.split(',')]
            cardiac_ids[disease_name] = ids_list
        except ValueError as e:
            print(f"Error parsing IDs for {disease_name}: {e}")
    
    return cardiac_ids

with open('./data/shriya/SHMOLLI-output-unet-myocardium/disease_patient_IDs.txt', 'r') as file:
    text_content = file.read()
    cardiac_ids_dict = parse_ids_to_dict(text_content)

In [ ]:
labelled_pheno_data = pd.read_csv("./data/bruna/lvedv/ukbb_all_no_exclusion_all_2_and_3_with_CMR_and_MR", sep='\t', header=0)

In [ ]:
type1_diabetes_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("E10", na=False)]["id"])
type2_diabetes_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("E11", na=False)]["id"])

MI_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I21", na=False)]["id"])
DCM_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I42", na=False)]["id"])

sarcoidosis_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("D86", na=False)]["id"])
chronic_kidney_disease_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("N18", na=False)]["id"])
hypertension_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I10", na=False)]["id"])

atherosclerosis_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I70", na=False)]["id"])
COPD_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("J44", na=False)]["id"])

In [ ]:
unet_t1_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')
latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_latent_dimentions_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')
cardiac_phenotype_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/cardiac_merged2.no_outliers.residuals.qnorm.txt" , sep='\t')

In [ ]:
unet_t1_regressed.replace(-1000, np.nan, inplace=True)
latent_dimensions.replace(-1000, np.nan, inplace=True)

unet_t1_regressed = unet_t1_regressed.dropna()
latent_dimensions = latent_dimensions.dropna()

unet_t1_regressed = unet_t1_regressed.drop(columns = ['Mean_T1_regressed_hematocrit',
                    'T1_Standard_Deviation_regressed_hematocrit',
                    'T1_0.25th_Percentile_regressed_hematocrit', 
                    'T1_1th_Percentile_regressed_hematocrit',
                    'T1_25th_Percentile_regressed_hematocrit',
                    'T1_50th_Percentile_regressed_hematocrit',
                    'T1_75th_Percentile_regressed_hematocrit',
                    'T1_99th_Percentile_regressed_hematocrit',
                    'T1_99.75th_Percentile_regressed_hematocrit',
                  'Mean_T1_regressed_hematocrit_hypertension_status',
                    'T1_Standard_Deviation_regressed_hematocrit_hypertension_status',
                    'T1_0.25th_Percentile_regressed_hematocrit_hypertension_status', 
                    'T1_1th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_25th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_50th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_75th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_99th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_99.75th_Percentile_regressed_hematocrit_hypertension_status'])

In [ ]:
valid_images = np.load("./data/shriya/SHMOLLI-output-unet-myocardium-update2/quality_ID_list.csv.npy").tolist()
bad_images = np.load("./data/shriya/SHMOLLI-output-unet-myocardium-update2/bad_ID_list.csv.npy").tolist()

In [ ]:
unet_t1_regressed = unet_t1_regressed[~unet_t1_regressed["IID"].isin(bad_images)]
latent_dimensions = latent_dimensions[~latent_dimensions["IID"].isin(bad_images)]
cardiac_phenotype_data = cardiac_phenotype_data[~cardiac_phenotype_data["id"].isin(bad_images)]

# Delta Rank Test

In [ ]:
def delta_rank_test(df, disease_ids, phenotype_name):
    print(f"Total patients in dataframe: {len(df)}")
    print(f"Number of patients with {phenotype_name}: {len(disease_ids)}")
    
    # Create a mask for patients with the disease
    mask = df['IID'].isin(disease_ids)
    control_mask = df['IID'].isin(no_disease_ids)
    
    # Extract values for patients with and without the disease
    disease_df = df[mask]
    #non_disease_df = df[~mask]
    non_disease_df = df[control_mask]
    
    value_columns = df.columns[2:]
    
    results_dict = {
        'phenotype': [],
        'disease_count': len(disease_df),
        'non_disease_count': len(non_disease_df),
        'disease_median': [],
        'non_disease_median': [],
        'p_value': [],
        'u_statistic': [],
        'effect_size': [],
        'Significant?': []
    }
    
    # Process each column separately
    for col in value_columns:
        disease_values = disease_df[col].dropna().values
        non_disease_values = non_disease_df[col].dropna().values
        
        # Skip if any group has no data
        if len(disease_values) == 0 or len(non_disease_values) == 0:
            continue
            
        # Perform rank-sum test (Mann-Whitney U test)
        u_stat, p_value = stats.mannwhitneyu(
            disease_values, 
            non_disease_values,
            alternative='two-sided'
        )
        
        # Calculate effect size (common language effect size)
        effect_size = 1 - (2 * u_stat) / (len(disease_values) * len(non_disease_values))
        
        results_dict['phenotype'].append(col)
        results_dict['disease_median'].append(np.median(disease_values))
        results_dict['non_disease_median'].append(np.median(non_disease_values))
        results_dict['p_value'].append(p_value)
        results_dict['u_statistic'].append(u_stat)
        results_dict['effect_size'].append(effect_size)
        results_dict['Significant?'].append(p_value < 0.00025)
    
    # Convert results to a DataFrame
    results_df = pd.DataFrame(results_dict)
    
    return results_df

In [ ]:
disease_groups = {
    'HCM': cardiac_ids_dict["HCM"],
    'Amyloidosis': cardiac_ids_dict["Amyloidosis"],
    'Ischemic': cardiac_ids_dict["Ischemic"], 
    'Non-Ischemic': cardiac_ids_dict["NonIschemic"], 
    'Type 1 Diabetes': type1_diabetes_patients,
    'Type 2 Diabetes': type2_diabetes_patients,
    'Myocardial Infarction': MI_patients,
    'Chronic Kidney Disease': chronic_kidney_disease_patients,
    'Hypertension': hypertension_patients,
    'Sarcoidosis': sarcoidosis_patients
}

In [ ]:
# Get all disease patient IDs combined
all_disease_ids = set()
for ids in disease_groups.values():
    all_disease_ids.update(ids)

# Get IIDs with no disease
no_disease_ids = set(unet_t1_regressed['IID']) - all_disease_ids

disease_groups['No Disease'] = no_disease_ids

In [ ]:
delta_rank_test(unet_t1_regressed, type1_diabetes_patients, "Type 1 Diabetes")

In [ ]:
delta_rank_test(unet_t1_regressed, type2_diabetes_patients, "Type 2 Diabetes")

In [ ]:
delta_rank_test(unet_t1_regressed, MI_patients, "myocardial infarction")

In [ ]:
delta_rank_test(unet_t1_regressed, hypertension_patients, "hypertension")

In [ ]:
delta_rank_test(unet_t1_regressed, cardiac_ids_dict["HCM"], "HCM")

In [ ]:
delta_rank_test(unet_t1_regressed, DCM_patients, "DCM")

In [ ]:
delta_rank_test(unet_t1_regressed, chronic_kidney_disease_patients, "Chronic Kidney Disease")

In [ ]:
delta_rank_test(unet_t1_regressed, sarcoidosis_patients, "Sarcoidosis")

In [ ]:
delta_rank_test(unet_t1_regressed, cardiac_ids_dict["Ischemic"], "Ischemic")

In [ ]:
delta_rank_test(unet_t1_regressed, cardiac_ids_dict["NonIschemic"], "Non-Ischemic")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import seaborn as sns

def plot_delta_rank_distribution(df, disease_ids, phenotype_col, disease_name, figsize=(12, 5)):
    """
    Visualize the distribution of a phenotype between disease and non-disease groups.
    
    Parameters:
        df: DataFrame containing IID and phenotype columns
        disease_ids: list/set of IIDs with the disease
        phenotype_col: column name to visualize
        disease_name: string label for the disease
        figsize: figure size tuple
    """
    mask = df['IID'].isin(disease_ids)
    disease_vals = df.loc[mask, phenotype_col].dropna().values
    non_disease_vals = df.loc[~mask, phenotype_col].dropna().values

    # Stats
    u_stat, p_value = stats.mannwhitneyu(disease_vals, non_disease_vals, alternative='two-sided')
    effect_size = 1 - (2 * u_stat) / (len(disease_vals) * len(non_disease_vals))

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(f'{phenotype_col} — {disease_name} vs Non-{disease_name}', 
                 fontsize=13, fontweight='bold', y=1.02)

    colors = {'disease': '#E3342F', 'non_disease': '#3490DC'}

    # --- KDE plot ---
    ax1 = axes[0]
    sns.kdeplot(non_disease_vals, ax=ax1, color=colors['non_disease'], 
                fill=True, alpha=0.3, linewidth=2, label=f'Non-{disease_name} (n={len(non_disease_vals):,})')
    sns.kdeplot(disease_vals, ax=ax1, color=colors['disease'], 
                fill=True, alpha=0.3, linewidth=2, label=f'{disease_name} (n={len(disease_vals):,})')

    ax1.axvline(np.median(disease_vals), color=colors['disease'], linestyle='--', linewidth=1.5, alpha=0.8)
    ax1.axvline(np.median(non_disease_vals), color=colors['non_disease'], linestyle='--', linewidth=1.5, alpha=0.8)

    ax1.set_xlabel(phenotype_col, fontsize=11)
    ax1.set_ylabel('Density', fontsize=11)
    ax1.set_title('Distribution (KDE)', fontsize=11)
    ax1.legend(fontsize=9)
    ax1.spines[['top', 'right']].set_visible(False)

    # Annotate stats
    sig_str = f'p = {p_value:.2e}\nEffect size = {effect_size:.3f}'
    ax1.text(0.97, 0.97, sig_str, transform=ax1.transAxes,
             fontsize=9, va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='grey', alpha=0.8))

    # --- Box + strip plot ---
    ax2 = axes[1]
    plot_data = pd.DataFrame({
        'value': np.concatenate([disease_vals, non_disease_vals]),
        'group': [disease_name] * len(disease_vals) + [f'Non-{disease_name}'] * len(non_disease_vals)
    })

    sns.boxplot(data=plot_data, x='group', y='value', ax=ax2,
                palette={disease_name: colors['disease'], f'Non-{disease_name}': colors['non_disease']},
                width=0.4, flierprops=dict(marker='o', markersize=2, alpha=0.3),
                boxprops=dict(alpha=0.7))

    # Overlay medians as text
    for i, (group, color) in enumerate([(disease_name, colors['disease']), 
                                         (f'Non-{disease_name}', colors['non_disease'])]):
        median = plot_data.loc[plot_data['group'] == group, 'value'].median()
        ax2.text(i, median, f'  {median:.3f}', va='center', fontsize=9, color=color, fontweight='bold')

    ax2.set_xlabel('')
    ax2.set_ylabel(phenotype_col, fontsize=11)
    ax2.set_title('Boxplot', fontsize=11)
    ax2.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.show()

    print(f"\nSummary — {phenotype_col} | {disease_name}")
    print(f"  Disease median:     {np.median(disease_vals):.4f}")
    print(f"  Non-disease median: {np.median(non_disease_vals):.4f}")
    print(f"  U statistic:        {u_stat:.1f}")
    print(f"  p-value:            {p_value:.2e}")
    print(f"  Effect size (r_rb): {effect_size:.4f}")

In [ ]:
# Single phenotype
plot_delta_rank_distribution(
    unet_t1_regressed, 
    cardiac_ids_dict["Ischemic"], 
    'T1_25th_Percentile', 
    'Ischemic'
)

In [ ]:
# Loop over all phenotypes
for col in unet_t1_regressed.columns[2:]:
    plot_delta_rank_distribution(unet_t1_regressed, MI_patients, col, 'Myocardial Infarction')

### Latent Dimensions Delta Rank

In [ ]:
delta_rank_test(latent_dimensions, type2_diabetes_patients, "Type 2 Diabetes")

In [ ]:
delta_rank_test(latent_dimensions, DCM_patients, "DCM")

In [ ]:
delta_rank_test(latent_dimensions, cardiac_ids_dict["HCM"], "HCM")

In [ ]:
delta_rank_test(latent_dimensions, hypertension_patients, "hypertension")

In [ ]:
delta_rank_test(latent_dimensions, MI_patients, "MI")

In [ ]:
delta_rank_test(latent_dimensions, sarcoidosis_patients, "sarcoidosis")

In [ ]:
delta_rank_test(latent_dimensions, cardiac_ids_dict["NonIschemic"], "NonIschemic")

In [ ]:
delta_rank_test(latent_dimensions, cardiac_ids_dict["Ischemic"], "Ischemic")

In [ ]:
delta_rank_test(latent_dimensions, chronic_kidney_disease_patients, "Chronic Kidney Disease")

In [ ]:
delta_rank_test(latent_dimensions, type1_diabetes_patients, "Type 1 Diabetes")

In [ ]:
delta_rank_test(latent_dimensions, MI_patients, "MI")

In [ ]:
t1_diseases = []
t1_disease_counts = []
t1_all_phenotypes = []
t1_all_effect_sizes = []
t1_all_p_values = []

t1_metrics = ['Mean_T1', 'T1_Standard_Deviation', 'T1_0.25th_Percentile', 'T1_1th_Percentile', 'T1_25th_Percentile',
           'T1_50th_Percentile', 'T1_75th_Percentile', 'T1_99th_Percentile', 'T1_99.75th_Percentile']


for disease, patient_IDs in disease_groups.items():
    result_df = delta_rank_test(unet_t1_regressed, patient_IDs, str(disease))
    t1_diseases.append(str(disease))
    t1_disease_counts.append(len(patient_IDs))  # From your output
    phenotypes = result_df['phenotype'].tolist()
    effect_sizes = result_df['effect_size'].tolist()
    p_values = result_df['p_value'].tolist()
        
    t1_all_phenotypes.append(phenotypes)
    t1_all_effect_sizes.append(effect_sizes)
    t1_all_p_values.append(p_values)

In [ ]:
latent_diseases = []
latent_disease_counts = []
latent_all_phenotypes = []
latent_all_effect_sizes = []
latent_all_p_values = []

for disease, patient_IDs in disease_groups.items():
    result_df = delta_rank_test(latent_dimensions, patient_IDs, str(disease))
    latent_diseases.append(str(disease))
    latent_disease_counts.append(len(patient_IDs))  # From your output
    phenotypes = result_df['phenotype'].tolist()
    effect_sizes = result_df['effect_size'].tolist()
    p_values = result_df['p_value'].tolist()
        
    latent_all_phenotypes.append(phenotypes)
    latent_all_effect_sizes.append(effect_sizes)
    latent_all_p_values.append(p_values)

In [ ]:
latent_all_phenotypes

In [ ]:
def create_delta_rank_effect_size_plot(diseases, metrics, effect_sizes, p_values, disease_counts):
    # Set up the figure with subplots - one subplot per disease
    fig, axes = plt.subplots(len(diseases), 1, figsize=(12, 3*len(diseases)), sharex=True)
    
    # If there's only one disease, make axes iterable
    if len(diseases) == 1:
        axes = [axes]
    
    # Define color palette for different metric categories
    t1_color = '#1f77b4'  # blue for T1 metrics
    vae_color = '#ff7f0e'  # orange for VAE latent dimensions
    
    # Track the overall max effect size for axis scaling
    overall_max_effect = 0
    
    # For each disease, create a horizontal forest plot
    for i, disease in enumerate(diseases):
        ax = axes[i]
        
        # Filter data for this disease
        disease_metrics = []
        disease_effects = []
        disease_p = []
        disease_colors = []
        metric_is_significant = []
        
        for j, metric in enumerate(metrics):
            # Check if effect size exists for this disease-metric combination
            if effect_sizes[i][j] is not None:
                disease_metrics.append(metric)
                disease_effects.append(effect_sizes[i][j])
                disease_p.append(p_values[i][j])
                
                # Determine color based on metric type
                if 'Latent' in metric:
                    disease_colors.append(vae_color)
                else:
                    disease_colors.append(t1_color)
                
                # Check significance (p < 0.00025 after Bonferroni correction)
                metric_is_significant.append(p_values[i][j] < 0.00025)
                
                # Update overall max effect size
                if abs(effect_sizes[i][j]) > overall_max_effect:
                    overall_max_effect = abs(effect_sizes[i][j])
        
        # Sort by absolute effect size
        sorted_indices = np.argsort(np.abs(disease_effects))
        disease_metrics = [disease_metrics[j] for j in sorted_indices]
        disease_effects = [disease_effects[j] for j in sorted_indices]
        disease_p = [disease_p[j] for j in sorted_indices]
        disease_colors = [disease_colors[j] for j in sorted_indices]
        metric_is_significant = [metric_is_significant[j] for j in sorted_indices]
        
        # Create horizontal bars for effect sizes
        y_pos = np.arange(len(disease_metrics))
        
        # Plot bars
        bars = ax.barh(y_pos, disease_effects, height=0.6, color=disease_colors, alpha=0.7)
        
        # Highlight significant metrics with a thicker edge
        for j, bar in enumerate(bars):
            if metric_is_significant[j]:
                bar.set_edgecolor('black')
                bar.set_linewidth(1.5)
        
        # Add metric names to y-axis
        ax.set_yticks(y_pos)
        ax.set_yticklabels(disease_metrics)
        
        # Add p-values as text only for significant values
        for j, effect in enumerate(disease_effects):
            # Only display p-values for statistically significant metrics
            if metric_is_significant[j]:
                # Format p-value text
                if disease_p[j] < 0.00001:
                    p_text = "p < 0.00001 ***"
                else:
                    p_text = f"p = {disease_p[j]:.6f} ***"
                
                # Position text at the end of the bar
                ax.text(effect + 0.02*overall_max_effect, j, p_text, va='center', ha='left', fontsize=8)
        
        # Add disease name and count
        ax.set_title(f"{disease} (n={disease_counts[i]})", fontweight='bold', fontsize=18)
        
        # Add grid lines
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        
        # Remove top and right spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Set x limits to start from 0 (since all effects are positive) with some padding
    for ax in axes:
        ax.set_xlim(0, overall_max_effect*1.3)  # Add 30% padding for text
    
    # Set common x-label
    fig.text(0.5, -0.01, 'Standardized Effect Size (Delta Rank)', ha='center', fontsize=12)
    
    # Add a overall title
    plt.suptitle('Delta Rank Analysis: Standardized Effect Sizes by Disease', fontsize=16, y=1.01)
    
    plt.tight_layout()
    return fig

In [ ]:
fig = create_delta_rank_effect_size_plot(t1_diseases, t1_metrics, t1_all_effect_sizes, t1_all_p_values, t1_disease_counts)

In [ ]:
fig = create_delta_rank_effect_size_plot(latent_diseases, latent_all_phenotypes[0], latent_all_effect_sizes, latent_all_p_values, latent_disease_counts)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Circle

def create_delta_rank_bubble_table(diseases, metrics, effect_sizes, p_values, disease_counts):
    """
    Creates a bubble table visualization where:
    - Rows = diseases
    - Columns = metrics
    - Color = effect size (beta)
    - Size = -log10(p-value)
    
    Parameters:
    -----------
    diseases : list
        List of disease names
    metrics : list
        List of metric names
    effect_sizes : 2D array/list
        Effect sizes [disease][metric]
    p_values : 2D array/list
        P-values [disease][metric]
    disease_counts : list
        Sample counts for each disease
    """
    
    # Set up figure
    fig, ax = plt.subplots(figsize=(max(14, len(metrics) * 1.2), max(8, len(diseases) * 1.2)))
    
    # Calculate -log10(p-values) for bubble sizes
    neg_log_p = []
    for i in range(len(diseases)):
        row = []
        for j in range(len(metrics)):
            # Check if this index exists in the effect_sizes for this disease
            if j < len(effect_sizes[i]) and effect_sizes[i][j] is not None:
                if p_values[i][j] is not None and p_values[i][j] > 0:
                    row.append(-np.log10(p_values[i][j]))
                else:
                    row.append(0)
            else:
                row.append(0)
        neg_log_p.append(row)
    
    # Flatten arrays for min/max calculations (excluding None values)
    valid_effects = []
    valid_neg_log_p = []
    
    for i in range(len(diseases)):
        for j in range(len(metrics)):
            # Only process if index exists in effect_sizes array
            if j < len(effect_sizes[i]) and effect_sizes[i][j] is not None:
                valid_effects.append(effect_sizes[i][j])
                valid_neg_log_p.append(neg_log_p[i][j])
    
    # Determine color scale range (symmetric around 0)
    max_effect = max(abs(min(valid_effects)), abs(max(valid_effects)))
    
    # Determine size scale range
    max_neg_log_p = max(valid_neg_log_p)
    min_neg_log_p = min(valid_neg_log_p)
    
    # Create colormap (white to blue for positive, white to red for negative)
    colors_negative = ['#d73027', '#fc8d59', '#fee090', '#ffffff']
    colors_positive = ['#ffffff', '#e0f3f8', '#91bfdb', '#4575b4']
    n_bins = 100
    
    # Create custom diverging colormap
    cmap = LinearSegmentedColormap.from_list('custom_diverging', 
                                             colors_negative + colors_positive[1:], 
                                             N=n_bins*2)
    norm = Normalize(vmin=-max_effect, vmax=max_effect)
    
    # Significance threshold after Bonferroni correction
    bonferroni_threshold = 0.00025
    significance_line = -np.log10(bonferroni_threshold)
    
    # Plot bubbles
    for i, disease in enumerate(diseases):
        for j, metric in enumerate(metrics):
            # Only plot if index exists in effect_sizes array
            if j < len(effect_sizes[i]) and effect_sizes[i][j] is not None:
                effect = effect_sizes[i][j]
                neg_log = neg_log_p[i][j]
                p_val = p_values[i][j]
                
                # Check if significant
                is_significant = neg_log >= significance_line
                
                # Determine bubble size
                if is_significant:
                    # For significant results, scale by -log p-value
                    if max_neg_log_p > min_neg_log_p:
                        size_scale = (neg_log - min_neg_log_p) / (max_neg_log_p - min_neg_log_p)
                    else:
                        size_scale = 0.5
                    radius = 0.15 + 0.35 * size_scale  # radius between 0.15 and 0.5
                else:
                    # For non-significant results, use small uniform size
                    radius = 0.12
                
                # Get color
                color = cmap(norm(effect))
                
                # Draw bubble
                circle = Circle((j, len(diseases) - 1 - i), radius, 
                               color=color, ec='gray', linewidth=0.5, zorder=2, alpha=0.8)
                ax.add_patch(circle)
                
                # Add edge highlight for significant results
                if is_significant:
                    circle.set_linewidth(2.5)
                    circle.set_edgecolor('black')
    
    # Set up grid
    ax.set_xlim(-0.8, len(metrics) + 3)  # Extended to accommodate legend
    ax.set_ylim(-0.8, len(diseases) - 0.2)
    
    # Add grid lines
    for i in range(len(diseases) + 1):
        ax.axhline(i - 0.5, color='gray', linestyle='-', linewidth=0.5, alpha=0.3, zorder=1)
    for j in range(len(metrics) + 1):
        ax.axvline(j - 0.5, color='gray', linestyle='-', linewidth=0.5, alpha=0.3, zorder=1)
    
    # Set ticks and labels
    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels(metrics, rotation=45, ha='right', fontsize=10)
    
    ax.set_yticks(range(len(diseases)))
    y_labels = [f"{disease} (n={disease_counts[i]})" for i, disease in enumerate(diseases)]
    ax.set_yticklabels(reversed(y_labels), fontsize=10)
    
    # Remove spines
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    # Remove tick marks
    ax.tick_params(left=False, bottom=False)
    
    # Add title
    ax.set_title('Delta Rank Analysis: Effect Sizes and Significance by Disease', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Add colorbar for effect sizes
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', pad=0.15, aspect=40, shrink=0.6)
    cbar.set_label('Standardized Effect Size (β)', fontsize=11, weight='bold')
    
    # Add size legend - only show range for significant results
    legend_x = len(metrics) + 1
    legend_y_start = len(diseases) - 2
    
    ax.text(legend_x, legend_y_start + 1.2, 'Bubble Size', 
            fontsize=11, fontweight='bold', ha='center')
    
    # Show 3 sizes for significant results
    significant_sizes = [max_neg_log_p, (min_neg_log_p + max_neg_log_p)/2, min_neg_log_p]
    
    for idx, log_val in enumerate(significant_sizes):
        y_pos = legend_y_start - idx * 0.7
        
        # Calculate radius for significant results
        if max_neg_log_p > min_neg_log_p:
            size_scale = (log_val - min_neg_log_p) / (max_neg_log_p - min_neg_log_p)
        else:
            size_scale = 0.5
        radius = 0.15 + 0.35 * size_scale
        
        # Draw legend circle
        circle = Circle((legend_x, y_pos), radius, 
                       color='lightgray', ec='black', linewidth=2, zorder=10)
        ax.add_patch(circle)
        
        # Convert -log10(p) back to p-value for display
        p_val = 10 ** (-log_val)
        
        # Format the label
        if p_val < 0.00001:
            label = f'p < 0.00001'
        elif p_val < 0.001:
            label = f'p < 0.001'
        else:
            label = f'p ≈ {p_val:.4f}'
        
        ax.text(legend_x + 0.6, y_pos, label, fontsize=9, va='center', ha='left')
    
    # Add non-significant bubble
    y_pos_nonsig = y_pos - 0.8
    circle = Circle((legend_x, y_pos_nonsig), 0.12, 
                   color='lightgray', ec='gray', linewidth=0.5, zorder=10)
    ax.add_patch(circle)
    ax.text(legend_x + 0.6, y_pos_nonsig, 'Not significant', fontsize=9, va='center', ha='left', style='italic')
    
    # Add note about significance threshold
    ax.text(legend_x, y_pos_nonsig - 0.7, 
            f'Bold edge: p < {bonferroni_threshold}', 
            fontsize=8, ha='center', style='italic', 
            bbox=dict(boxstyle='round,pad=0.4', facecolor='yellow', alpha=0.3, edgecolor='black'))
    
    plt.tight_layout()
    return fig

In [ ]:
# Properly combine data for each disease
combined_metrics = t1_metrics + latent_all_phenotypes[0]

combined_effect_sizes = []
combined_p_values = []

for i in range(len(t1_diseases)):  # Assuming same diseases in both
    # Combine effect sizes for this disease
    disease_effects = list(t1_all_effect_sizes[i]) + list(latent_all_effect_sizes[i])
    combined_effect_sizes.append(disease_effects)
    
    # Combine p-values for this disease
    disease_pvals = list(t1_all_p_values[i]) + list(latent_all_p_values[i])
    combined_p_values.append(disease_pvals)

combined_disease_counts = t1_disease_counts  # Assuming same counts

# Now call the function
create_delta_rank_bubble_table(
    t1_diseases,
    combined_metrics,
    combined_effect_sizes,
    combined_p_values,
    combined_disease_counts
)

In [ ]:
t1_all_p_values + latent_all_p_values

# Spacial Clustering

In [ ]:
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
latent_cluster_data = latent_dimensions.copy()

latent_cluster_data = latent_cluster_data.drop(columns=['FID', "IID"])

In [ ]:
data_transposed = latent_cluster_data.T
column_names = latent_cluster_data.columns


# Standardize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_transposed)
data_scaled_df = pd.DataFrame(data_scaled, index=data_transposed.index)

# Compute distance matrix between dimensions
dist_matrix = pdist(data_scaled_df, metric='euclidean')

# Create linkage matrix
linkage_matrix = hierarchy.linkage(dist_matrix, method='ward')

# Plot dendrogram to visualize hierarchical clustering
plt.figure(figsize=(12, 6))
plt.title('Hierarchical Clustering of Latent Dimensions', fontsize=16)
dendrogram = hierarchy.dendrogram(
    linkage_matrix,
    labels=column_names,
    leaf_font_size=12,
    color_threshold=0.7 * max(linkage_matrix[:, 2])
)
plt.xlabel('Latent Dimensions', fontsize=14)
plt.ylabel('Distance', fontsize=14)
plt.tight_layout()
plt.savefig('latent_dimensions_dendrogram.png')
plt.close()

# Determine the optimal number of clusters using silhouette score
silhouette_scores = []
range_n_clusters = range(2, min(10, len(column_names)))

for n_clusters in range_n_clusters:
    clusterer = AgglomerativeClustering(n_clusters=n_clusters)
    cluster_labels = clusterer.fit_predict(data_scaled_df)
    silhouette_avg = silhouette_score(data_scaled_df, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"For n_clusters = {n_clusters}, the silhouette score is {silhouette_avg:.3f}")

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(range_n_clusters, silhouette_scores, 'o-', linewidth=2, markersize=8)
plt.title('Silhouette Score for Different Numbers of Clusters', fontsize=16)
plt.xlabel('Number of Clusters', fontsize=14)
plt.ylabel('Silhouette Score', fontsize=14)
plt.grid(True)
plt.savefig('silhouette_scores.png')
plt.close()

# Choose the number of clusters with highest silhouette score
optimal_clusters = range_n_clusters[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters: {optimal_clusters}")

# Apply hierarchical clustering with the optimal number of clusters
hierarchical = AgglomerativeClustering(n_clusters=optimal_clusters)
clusters = hierarchical.fit_predict(data_scaled_df)

# Add cluster labels to the dataframe
data_transposed['Cluster'] = clusters

# Print the clusters
print("\nClusters of latent dimensions:")
for cluster_id in range(optimal_clusters):
    dimensions_in_cluster = data_transposed[data_transposed['Cluster'] == cluster_id].index.tolist()
    print(f"Cluster {cluster_id + 1}: {dimensions_in_cluster}")

# Create a heatmap to visualize relationships between latent dimensions
plt.figure(figsize=(12, 10))
correlation_matrix = latent_cluster_data.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', 
            fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation between Latent Dimensions', fontsize=16)
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.close()

In [ ]:
# Standardize the data for samples
scaler_samples = StandardScaler()
samples_scaled = scaler_samples.fit_transform(latent_cluster_data)
samples_scaled_df = pd.DataFrame(samples_scaled, index=latent_cluster_data.index, columns=latent_cluster_data.columns)

# Compute distance matrix between samples
sample_dist_matrix = pdist(samples_scaled_df, metric='euclidean')

# Create linkage matrix for samples
sample_linkage_matrix = hierarchy.linkage(sample_dist_matrix, method='ward')

# Plot dendrogram for patient clustering
plt.figure(figsize=(10, 6))
plt.title('Hierarchical Clustering of Patients', fontsize=16)
sample_dendrogram = hierarchy.dendrogram(
    sample_linkage_matrix,
    labels=latent_cluster_data.index,
    leaf_font_size=12,
    color_threshold=0.7 * max(sample_linkage_matrix[:, 2])
)
plt.xlabel('Patients', fontsize=14)
plt.ylabel('Distance', fontsize=14)
plt.tight_layout()
plt.savefig('patient_clustering_dendrogram.png')
plt.close()

# Determine optimal number of patient clusters
sample_silhouette_scores = []
sample_range_n_clusters = range(2, min(6, len(patient_ids)))

for n_clusters in sample_range_n_clusters:
    clusterer = AgglomerativeClustering(n_clusters=n_clusters)
    cluster_labels = clusterer.fit_predict(samples_scaled_df)
    silhouette_avg = silhouette_score(samples_scaled_df, cluster_labels)
    sample_silhouette_scores.append(silhouette_avg)
    print(f"For patient clusters = {n_clusters}, the silhouette score is {silhouette_avg:.3f}")

# Choose optimal number of patient clusters
optimal_patient_clusters = sample_range_n_clusters[np.argmax(sample_silhouette_scores)]
print(f"Optimal number of patient clusters: {optimal_patient_clusters}")

# Apply clustering with optimal number of clusters
patient_hierarchical = AgglomerativeClustering(n_clusters=optimal_patient_clusters)
patient_clusters = patient_hierarchical.fit_predict(samples_scaled_df)

# Create results dataframe
results = data.copy()
results['Cluster'] = patient_clusters
print("\nPatient clustering results:")
print(results[['Cluster']].head(10))

# Visualize patient clusters in 2D (if you want to use dimensionality reduction)
from sklearn.decomposition import PCA

# Apply PCA to reduce to 2 dimensions for visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(samples_scaled_df)

# Create a dataframe for the PCA results
pca_df = pd.DataFrame(data=pca_result, columns=['PC1', 'PC2'], index=patient_ids)
pca_df['Cluster'] = patient_clusters

# Plot the clusters
plt.figure(figsize=(10, 8))
sns.scatterplot(x='PC1', y='PC2', hue='Cluster', data=pca_df, palette='viridis', s=100)
for idx in pca_df.index:
    plt.annotate(idx, (pca_df.loc[idx, 'PC1'], pca_df.loc[idx, 'PC2']), 
                 fontsize=9, alpha=0.7)
plt.title('Patient Clusters based on Latent Dimensions', fontsize=16)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=14)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=14)
plt.legend(title='Cluster')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# T1 Descriptive Statistics

In [ ]:
big_pheno_data = pd.read_csv("./data/bruna/lvedv/ukbb_all_no_exclusion_all_2_and_3_with_CMR_and_MR", sep='\t', header=0)

In [ ]:
unfiltered_unet_t1_raw = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/percentiles_T1.csv")
unfiltered_unet_t1_raw["Patient_ID"] = [int(x[:7]) for x in unfiltered_unet_t1_raw["Patient_ID"]]

genetic_filtered_unet_t1_raw = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed.csv_phenotypes_trimmed.txt", sep='\t')

valid_images = np.load("./data/shriya/SHMOLLI-output-unet-myocardium-update2/quality_ID_list.csv.npy").tolist()
valid_images = [int(x) for x in valid_images]

In [ ]:
patient_mapping = pd.read_table("./data/shriya/ukb22282_24983_mapping.tsv", header = None)

# Function to get the value from column 0 based on the ID in column 1
def get_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[1].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[1] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return mapping_df.loc[row_index, 0]
    else:
        return None  # Return None if the ID is not found

# Function to get the value from column 0 based on the ID in column 1
def reverse_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[0].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[0] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return int(mapping_df.loc[row_index, 1])
    else:
        return None  # Return None if the ID is not found

In [ ]:
all_UKBB_patient_IDs = sorted(big_pheno_data["id"])
all_imaged_patient_IDs = sorted(unfiltered_unet_t1_raw["Patient_ID"])
valid_images_IDs = valid_images

In [ ]:
#All Imaged Patients

# Filter big_pheno_data to only include rows where id is in valid_ids
filtered_data = big_pheno_data[big_pheno_data["id"].isin(all_imaged_patient_IDs)]

# Calculate mean and standard deviation of age
mean_age = filtered_data["age (y)"].mean()
std_age = filtered_data["age (y)"].std()

# Calculate percentage of "Male" in sex
male_percentage = (filtered_data["sex"] == "Male").mean() * 100

print(f"Total number: ", len(filtered_data))
print(f"Mean age: {mean_age:.2f} years")
print(f"Standard deviation of age: {std_age:.2f} years")
print(f"Percentage of Males: {male_percentage:.2f}%")

In [ ]:
# All Valid Patients

# Filter big_pheno_data to only include rows where id is in valid_images
filtered_data = big_pheno_data[big_pheno_data["id"].isin(valid_images)]

# Calculate mean and standard deviation of age
mean_age = filtered_data["age (y)"].mean()
std_age = filtered_data["age (y)"].std()

# Calculate percentage of "Male" in sex
male_percentage = (filtered_data["sex"] == "Male").mean() * 100

print(f"Total number: ", len(filtered_data))
print(f"Mean age: {mean_age:.2f} years")
print(f"Standard deviation of age: {std_age:.2f} years")
print(f"Percentage of Males: {male_percentage:.2f}%")

In [ ]:
filtered_data = unfiltered_unet_t1_raw[unfiltered_unet_t1_raw["Patient_ID"].isin(valid_images)]
print(f"Total number: ", len(unfiltered_unet_t1_raw))
print(f"Total valid: ", len(filtered_data))
print(f"Percentage: ", len(filtered_data)/len(unfiltered_unet_t1_raw) * 100)

In [ ]:
filtered_data = genetic_filtered_unet_t1_raw[genetic_filtered_unet_t1_raw["IID"].isin(valid_images)]
print(f"Total number: ", len(genetic_filtered_unet_t1_raw))
print(f"Total valid: ", len(filtered_data))
print(f"Percentage: ", len(filtered_data)/len(genetic_filtered_unet_t1_raw) * 100)

In [ ]:
import numpy as np
from scipy import stats

unet_t1_raw = unfiltered_unet_t1_raw[unfiltered_unet_t1_raw["IID"].isin(valid_images)]

# Calculate the basic statistics
mean_t1 = unet_t1_raw['Mean_T1'].mean()
std_t1 = unet_t1_raw['Mean_T1'].std()
q1 = unet_t1_raw['T1_25th_Percentile'].mean()
q3 = unet_t1_raw['T1_75th_Percentile'].mean()
skewness = stats.skew(unet_t1_raw['Mean_T1'])

# Calculate within-subject heterogeneity
avg_within_subject_std = unet_t1_raw['T1_Standard_Deviation'].mean()
std_within_subject_std = unet_t1_raw['T1_Standard_Deviation'].std()

# Get percentile values
p1_mean = unet_t1_raw['T1_1th_Percentile'].mean()
p1_std = unet_t1_raw['T1_1th_Percentile'].std()
p99_mean = unet_t1_raw['T1_99th_Percentile'].mean()
p99_std = unet_t1_raw['T1_99th_Percentile'].std()

# Calculate the percentage of successful segmentations
# Assuming total_initial_images is a variable you have that contains the total count
# of images in your initial dataset
total_initial_images = len(unfiltered_unet_t1_raw)  # Replace with your actual count if different
total_genetic_overap_images = len(unfiltered_unet_t1_raw)
success_images = len(unet_t1_raw)  # Replace with your actual count if different
success_percentage = (success_images/total_initial_images)*100.0  # Assuming all rows in unet_t1_raw are successful segmentations

# Print the results in the format you want
print("Results:")
print(f"Of total {total_genetic_overap_images} ShMOLLI images, {total_genetic_overap_images} patients also had genetic data avaliable.")
print(f"Automated segmentation of the myocardium was successfully performed on {len(unet_t1_raw):,} ShMOLLI images, "
      f"representing {success_percentage:.1f}% of the initial dataset.")

print(f"The mean T1 value across all subjects was {mean_t1:.1f} ± {std_t1:.1f} ms (mean ± SD), "
      f"with an interquartile range of {q1:.1f} ms to {q3:.1f} ms. "
      f"The overall distribution demonstrated {'positive' if skewness > 0 else 'negative'} skewness "
      f"(skewness coefficient = {skewness:.2f}), indicating a "
      f"{'longer tail toward higher' if skewness > 0 else 'longer tail toward lower'} T1 values.")

print(f"When examining the within-subject heterogeneity of T1 values, the average standard deviation of T1 "
      f"within individual myocardia was {avg_within_subject_std:.1f} ± {std_within_subject_std:.1f} ms, "
      f"suggesting substantial spatial variability in tissue characteristics even within normal hearts. "
      f"The 1st percentile and 99th percentile values within subjects had means of "
      f"{p1_mean:.1f} ± {p1_std:.1f} ms and {p99_mean:.1f} ± {p99_std:.1f} ms, respectively.")

In [ ]:
filtered_data = unfiltered_unet_t1_raw[unfiltered_unet_t1_raw["Patient_ID"].isin(valid_images)]
print(f"Total number: ", len(unfiltered_unet_t1_raw))
print(f"Total valid: ", len(filtered_data))
print(f"Percentage: ", len(filtered_data)/len(unfiltered_unet_t1_raw) * 100)

for col in filtered_data.columns:
    print(col)
    print(f"Mean: ", filtered_data[col].mean())
    print(f"STD: ", filtered_data[col].std())

In [ ]:
# Generate comprehensive descriptive statistics
descriptive_stats = filtered_data[filtered_data["T1_Standard_Deviation"] != 0].describe().T

# Or for more control:
stats_table = filtered_data.agg(['mean', 'std', 'median', 
                                  lambda x: x.quantile(0.25), 
                                  lambda x: x.quantile(0.75),
                                  'min', 'max']).T

In [ ]:
descriptive_stats = descriptive_stats.iloc[1:]

In [ ]:
filtered_data[filtered_data["T1_Standard_Deviation"] == 0]

In [ ]:
merged_data = pd.merge(unfiltered_unet_t1_raw, big_pheno_data, left_on = "Patient_ID", right_on = "id", how="inner")

In [ ]:
import pandas as pd

# Assuming you already ran the comprehensive analysis and have results_df
# If not, here's the full code:

t1_metrics = ['Mean_T1', 'T1_Standard_Deviation', 
              'T1_0.25th_Percentile', 'T1_1th_Percentile', 
              'T1_25th_Percentile', 'T1_50th_Percentile', 
              'T1_75th_Percentile', 'T1_99th_Percentile', 
              'T1_99.75th_Percentile']

results_summary = []

for metric in t1_metrics:
    # Females
    females_clean = females_df[[metric, 'age_decades']].dropna()
    if len(females_clean) > 0:
        slope_f, intercept_f, r_f, p_f, se_f = stats.linregress(
            females_clean['age_decades'], females_clean[metric])
        
        # Males
        males_clean = males_df[[metric, 'age_decades']].dropna()
        slope_m, intercept_m, r_m, p_m, se_m = stats.linregress(
            males_clean['age_decades'], males_clean[metric])
        
        # Calculate interaction p-value
        se_diff = np.sqrt(se_f**2 + se_m**2)
        z_stat = (slope_m - slope_f) / se_diff
        p_interaction = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        
        results_summary.append({
            'Metric': metric,
            'Female_N': len(females_clean),
            'Female_Beta': slope_f,
            'Female_SE': se_f,
            'Female_P': p_f,
            'Male_N': len(males_clean),
            'Male_Beta': slope_m,
            'Male_SE': se_m,
            'Male_P': p_m,
            'Sex_Difference': slope_m - slope_f,
            'P_Interaction': p_interaction
        })

results_df = pd.DataFrame(results_summary)

# Create publication-ready supplementary table
supp_table = pd.DataFrame()

# Clean up metric names
supp_table['T1 Metric'] = results_df['Metric'].str.replace('T1_', '').str.replace('_', ' ')

# Format Female results
supp_table['Female N'] = results_df['Female_N']
supp_table['Female β ± SE (ms/10 yr)'] = results_df.apply(
    lambda x: f"{x['Female_Beta']:.2f} ± {x['Female_SE']:.2f}", axis=1)
supp_table['Female P-value'] = results_df['Female_P'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

# Format Male results
supp_table['Male N'] = results_df['Male_N']
supp_table['Male β ± SE (ms/10 yr)'] = results_df.apply(
    lambda x: f"{x['Male_Beta']:.2f} ± {x['Male_SE']:.2f}", axis=1)
supp_table['Male P-value'] = results_df['Male_P'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

# Format interaction
supp_table['Sex Difference (ms/10 yr)'] = results_df['Sex_Difference'].apply(
    lambda x: f"{x:.2f}")
supp_table['P for Interaction'] = results_df['P_Interaction'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

print("\n=== SUPPLEMENTARY TABLE: SEX-SPECIFIC AGE ASSOCIATIONS WITH T1 METRICS ===\n")
print(supp_table.to_string(index=False))

# Save to CSV
supp_table.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/supplementary_table_age_sex_T1.csv', index=False)
print("\n✓ Table saved to: supplementary_table_age_sex_T1.csv")

In [ ]:
# ==== COMPREHENSIVE SUPPLEMENTARY TABLE ====

# Part 1: Sample Characteristics
sample_chars = pd.DataFrame({
    'Sex': ['Female', 'Male'],
    'N': [len(females_df), len(males_df)],
    'Age (years), mean ± SD': [
        f"{females_df['age (y)'].mean():.1f} ± {females_df['age (y)'].std():.1f}",
        f"{males_df['age (y)'].mean():.1f} ± {males_df['age (y)'].std():.1f}"
    ],
    'Age range': [
        f"{females_df['age (y)'].min():.0f}-{females_df['age (y)'].max():.0f}",
        f"{males_df['age (y)'].min():.0f}-{males_df['age (y)'].max():.0f}"
    ],
    'BMI (kg/m²), mean ± SD': [
        f"{females_df['BMI'].mean():.1f} ± {females_df['BMI'].std():.1f}",
        f"{males_df['BMI'].mean():.1f} ± {males_df['BMI'].std():.1f}"
    ]
})

print("=== SUPPLEMENTARY TABLE Part A: Sample Characteristics ===")
print(sample_chars.to_string(index=False))
print()

# Part 2: Comprehensive T1 Results
comprehensive_results = []

for metric in t1_metrics:
    # Females
    females_clean = females_df[[metric, 'age_decades']].dropna()
    slope_f, intercept_f, r_f, p_f, se_f = stats.linregress(
        females_clean['age_decades'], females_clean[metric])
    ci_lower_f = slope_f - 1.96 * se_f
    ci_upper_f = slope_f + 1.96 * se_f
    r2_f = r_f**2
    
    # Males
    males_clean = males_df[[metric, 'age_decades']].dropna()
    slope_m, intercept_m, r_m, p_m, se_m = stats.linregress(
        males_clean['age_decades'], males_clean[metric])
    ci_lower_m = slope_m - 1.96 * se_m
    ci_upper_m = slope_m + 1.96 * se_m
    r2_m = r_m**2
    
    # Interaction test
    se_diff = np.sqrt(se_f**2 + se_m**2)
    z_stat = (slope_m - slope_f) / se_diff
    p_interaction = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    # Mean T1 values by sex
    mean_f = females_df[metric].mean()
    sd_f = females_df[metric].std()
    mean_m = males_df[metric].mean()
    sd_m = males_df[metric].std()
    
    # T-test for sex difference in T1
    t_stat, p_sex = stats.ttest_ind(
        females_df[metric].dropna(), 
        males_df[metric].dropna()
    )
    
    comprehensive_results.append({
        'Metric': metric,
        # Descriptive stats
        'Female_Mean_T1': mean_f,
        'Female_SD_T1': sd_f,
        'Male_Mean_T1': mean_m,
        'Male_SD_T1': sd_m,
        'P_Sex_Diff': p_sex,
        # Female regression
        'Female_N': len(females_clean),
        'Female_Beta': slope_f,
        'Female_SE': se_f,
        'Female_CI_Lower': ci_lower_f,
        'Female_CI_Upper': ci_upper_f,
        'Female_P': p_f,
        'Female_R2': r2_f,
        # Male regression
        'Male_N': len(males_clean),
        'Male_Beta': slope_m,
        'Male_SE': se_m,
        'Male_CI_Lower': ci_lower_m,
        'Male_CI_Upper': ci_upper_m,
        'Male_P': p_m,
        'Male_R2': r2_m,
        # Interaction
        'Beta_Difference': slope_m - slope_f,
        'P_Interaction': p_interaction
    })

results_comprehensive = pd.DataFrame(comprehensive_results)

# Format for publication
supp_table_full = pd.DataFrame()

# Metric names
supp_table_full['T1 Metric'] = results_comprehensive['Metric'].str.replace('T1_', '').str.replace('_', ' ')

# Descriptive statistics
supp_table_full['Female Mean T1 ± SD (ms)'] = results_comprehensive.apply(
    lambda x: f"{x['Female_Mean_T1']:.1f} ± {x['Female_SD_T1']:.1f}", axis=1)
supp_table_full['Male Mean T1 ± SD (ms)'] = results_comprehensive.apply(
    lambda x: f"{x['Male_Mean_T1']:.1f} ± {x['Male_SD_T1']:.1f}", axis=1)
supp_table_full['P (Sex Difference)'] = results_comprehensive['P_Sex_Diff'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

# Female age associations
supp_table_full['Female β (95% CI)'] = results_comprehensive.apply(
    lambda x: f"{x['Female_Beta']:.2f} ({x['Female_CI_Lower']:.2f}, {x['Female_CI_Upper']:.2f})", axis=1)
supp_table_full['Female P'] = results_comprehensive['Female_P'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")
supp_table_full['Female R²'] = results_comprehensive['Female_R2'].apply(lambda x: f"{x:.4f}")

# Male age associations
supp_table_full['Male β (95% CI)'] = results_comprehensive.apply(
    lambda x: f"{x['Male_Beta']:.2f} ({x['Male_CI_Lower']:.2f}, {x['Male_CI_Upper']:.2f})", axis=1)
supp_table_full['Male P'] = results_comprehensive['Male_P'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")
supp_table_full['Male R²'] = results_comprehensive['Male_R2'].apply(lambda x: f"{x:.4f}")

# Interaction
supp_table_full['Sex × Age Interaction P'] = results_comprehensive['P_Interaction'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

print("\n=== SUPPLEMENTARY TABLE Part B: T1 Metrics and Age Associations ===")
supp_table_full

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate BMI
merged_data['BMI'] = merged_data['weight (kg)'] / (merged_data['height (cm)'] / 100) ** 2
merged_data['age_decades'] = merged_data['age (y)'] / 10

# Check sex encoding
print(merged_data['sex'].value_counts())
merged_data['sex_numeric'] = (merged_data['sex'].str.lower().isin(['m', 'male', '1'])).astype(int)

# Separate by sex
females_df = merged_data[merged_data['sex_numeric'] == 0].dropna(subset=['Mean_T1', 'age (y)', 'BMI'])
males_df = merged_data[merged_data['sex_numeric'] == 1].dropna(subset=['Mean_T1', 'age (y)', 'BMI'])

# SIMPLE LINEAR REGRESSION for Age effect
# Females
slope_f, intercept_f, r_f, p_f, se_f = stats.linregress(females_df['age_decades'], females_df['Mean_T1'])
print("\n=== FEMALES ===")
print(f"ΔT1(ms)/10 years = {slope_f:.2f} ± {se_f:.2f}")
print(f"P = {p_f:.2e}")
print(f"R² = {r_f**2:.3f}")

# Males
slope_m, intercept_m, r_m, p_m, se_m = stats.linregress(males_df['age_decades'], males_df['Mean_T1'])
print("\n=== MALES ===")
print(f"ΔT1(ms)/10 years = {slope_m:.2f} ± {se_m:.2f}")
print(f"P = {p_m:.2e}")
print(f"R² = {r_m**2:.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy import stats as sp_stats

# Females - multivariable
X_f = females_df[['age_decades', 'BMI', 'iLVM (g/m2)']].dropna()
y_f = females_df.loc[X_f.index, 'Mean_T1']

model_f = LinearRegression()
model_f.fit(X_f, y_f)

# Get p-values (requires manual calculation)
predictions_f = model_f.predict(X_f)
residuals_f = y_f - predictions_f
mse_f = np.sum(residuals_f**2) / (len(y_f) - X_f.shape[1] - 1)

# Standard error for age coefficient
var_coef_f = mse_f * np.linalg.inv(X_f.T.dot(X_f)).diagonal()
se_age_f = np.sqrt(var_coef_f[0])
t_stat_f = model_f.coef_[0] / se_age_f
p_value_f = 2 * (1 - sp_stats.t.cdf(abs(t_stat_f), len(y_f) - X_f.shape[1] - 1))

print("\n=== FEMALES (Adjusted) ===")
print(f"ΔT1(ms)/10 years = {model_f.coef_[0]:.2f} ± {se_age_f:.2f}")
print(f"P = {p_value_f:.2e}")
print(f"R² = {r2_score(y_f, predictions_f):.3f}")

# Repeat for males
X_m = males_df[['age_decades', 'BMI', 'iLVM (g/m2)']].dropna()
y_m = males_df.loc[X_m.index, 'Mean_T1']
model_m = LinearRegression()
model_m.fit(X_m, y_m)

predictions_m = model_m.predict(X_m)
residuals_m = y_m - predictions_m
mse_m = np.sum(residuals_m**2) / (len(y_m) - X_m.shape[1] - 1)
var_coef_m = mse_m * np.linalg.inv(X_m.T.dot(X_m)).diagonal()
se_age_m = np.sqrt(var_coef_m[0])
t_stat_m = model_m.coef_[0] / se_age_m
p_value_m = 2 * (1 - sp_stats.t.cdf(abs(t_stat_m), len(y_m) - X_m.shape[1] - 1))

print("\n=== MALES (Adjusted) ===")
print(f"ΔT1(ms)/10 years = {model_m.coef_[0]:.2f} ± {se_age_m:.2f}")
print(f"P = {p_value_m:.2e}")
print(f"R² = {r2_score(y_m, predictions_m):.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

# Females
ax.scatter(females_df['age (y)'], females_df['Mean_T1'], 
           alpha=0.3, s=10, color='#d62728', label='Female', rasterized=True)
age_range_f = np.array([females_df['age (y)'].min(), females_df['age (y)'].max()])
t1_pred_f = intercept_f + slope_f * (age_range_f / 10)
ax.plot(age_range_f, t1_pred_f, 'r-', linewidth=3, 
        label=f'Female: {slope_f:.2f} ± {se_f:.2f} ms/10yr (p={p_f:.2e})')

# Males
ax.scatter(males_df['age (y)'], males_df['Mean_T1'], 
           alpha=0.3, s=10, color='#1f77b4', label='Male', rasterized=True)
age_range_m = np.array([males_df['age (y)'].min(), males_df['age (y)'].max()])
t1_pred_m = intercept_m + slope_m * (age_range_m / 10)
ax.plot(age_range_m, t1_pred_m, 'b-', linewidth=3,
        label=f'Male: {slope_m:.2f} ± {se_m:.2f} ms/10yr (p={p_m:.2e})')

ax.set_xlabel('Age (years)', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean T1 (ms)', fontsize=14, fontweight='bold')
ax.set_title('Sex-Specific Age-Related T1 Changes', fontsize=16, fontweight='bold')
ax.legend(loc='best', fontsize=11, frameon=True, fancybox=True, shadow=True)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('age_T1_sex_specific.png', dpi=300, bbox_inches='tight')
plt.savefig('age_T1_sex_specific.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy import stats as sp_stats

def multivariable_regression(df, predictors, outcome):
    """Run multivariable regression and return formatted results"""
    # Drop missing values
    data_clean = df[predictors + [outcome]].dropna()
    X = data_clean[predictors]
    y = data_clean[outcome]
    
    # Fit model
    model = LinearRegression()
    model.fit(X, y)
    
    # Calculate statistics
    predictions = model.predict(X)
    residuals = y - predictions
    n = len(y)
    p = X.shape[1]
    mse = np.sum(residuals**2) / (n - p - 1)
    
    # Standard errors and p-values
    var_coef = mse * np.linalg.inv(X.T.dot(X)).diagonal()
    se_coef = np.sqrt(var_coef)
    t_stats = model.coef_ / se_coef
    p_values = 2 * (1 - sp_stats.t.cdf(abs(t_stats), n - p - 1))
    
    # R-squared
    r2 = 1 - (np.sum(residuals**2) / np.sum((y - y.mean())**2))
    
    return {
        'coef': model.coef_,
        'se': se_coef,
        'p_values': p_values,
        'r2': r2,
        'n': n
    }

# Females - adjusted analysis
predictors_f = ['age_decades', 'BMI', 'iLVM (g/m2)']
results_f_adj = multivariable_regression(females_df, predictors_f, 'Mean_T1')

print("\n=== FEMALES (Adjusted for BMI and iLVM) ===")
print(f"N = {results_f_adj['n']}")
print(f"ΔT1(ms)/10 years = {results_f_adj['coef'][0]:.2f} ± {results_f_adj['se'][0]:.2f}")
print(f"P = {results_f_adj['p_values'][0]:.2e}")
print(f"R² = {results_f_adj['r2']:.3f}")

# Males - adjusted analysis
predictors_m = ['age_decades', 'BMI', 'iLVM (g/m2)']
results_m_adj = multivariable_regression(males_df, predictors_m, 'Mean_T1')

print("\n=== MALES (Adjusted for BMI and iLVM) ===")
print(f"N = {results_m_adj['n']}")
print(f"ΔT1(ms)/10 years = {results_m_adj['coef'][0]:.2f} ± {results_m_adj['se'][0]:.2f}")
print(f"P = {results_m_adj['p_values'][0]:.2e}")
print(f"R² = {results_m_adj['r2']:.3f}")

In [ ]:
# Model 1: Age + Sex + BMI
merged_clean = merged_data[['Mean_T1', 'age_decades', 'sex_numeric', 'BMI']].dropna()
X1 = merged_clean[['age_decades', 'sex_numeric', 'BMI']]
y1 = merged_clean['Mean_T1']
model1 = LinearRegression().fit(X1, y1)
r2_basic = model1.score(X1, y1)

# Model 2: Add iLVM
merged_clean2 = merged_data[['Mean_T1', 'age_decades', 'sex_numeric', 'BMI', 'iLVM (g/m2)']].dropna()
X2 = merged_clean2[['age_decades', 'sex_numeric', 'BMI', 'iLVM (g/m2)']]
y2 = merged_clean2['Mean_T1']
model2 = LinearRegression().fit(X2, y2)
r2_cardiac = model2.score(X2, y2)

print("\n=== VARIANCE EXPLAINED ===")
print(f"Age + Sex + BMI: R² = {r2_basic:.4f}")
print(f"+ iLVM: R² = {r2_cardiac:.4f}")
print(f"Incremental R² from iLVM: {r2_cardiac - r2_basic:.4f}")

In [ ]:
t1_metrics = ['Mean_T1', 'T1_Standard_Deviation', 
              'T1_0.25th_Percentile', 'T1_1th_Percentile', 
              'T1_25th_Percentile', 'T1_50th_Percentile', 
              'T1_75th_Percentile', 'T1_99th_Percentile', 
              'T1_99.75th_Percentile']

results_summary = []

for metric in t1_metrics:
    # Females
    females_clean = females_df[[metric, 'age_decades']].dropna()
    if len(females_clean) > 0:
        slope_f, intercept_f, r_f, p_f, se_f = stats.linregress(
            females_clean['age_decades'], females_clean[metric])
        
        # Males
        males_clean = males_df[[metric, 'age_decades']].dropna()
        slope_m, intercept_m, r_m, p_m, se_m = stats.linregress(
            males_clean['age_decades'], males_clean[metric])
        
        results_summary.append({
            'Metric': metric,
            'Female_Beta': slope_f,
            'Female_SE': se_f,
            'Female_P': p_f,
            'Male_Beta': slope_m,
            'Male_SE': se_m,
            'Male_P': p_m,
            'Sex_Difference': slope_m - slope_f  # Magnitude of sex difference
        })

# Create comprehensive results table
results_df = pd.DataFrame(results_summary)
print("\n=== AGE ASSOCIATIONS ACROSS ALL T1 METRICS ===")
print(results_df.to_string(index=False))

In [ ]:
# Test if the age effect differs significantly between sexes for each metric
from scipy.stats import ttest_ind_from_stats

print("\n=== TEST FOR SEX DIFFERENCES IN AGE EFFECTS ===")
for idx, row in results_df.iterrows():
    # Z-test for difference in slopes
    se_diff = np.sqrt(row['Female_SE']**2 + row['Male_SE']**2)
    z_stat = (row['Male_Beta'] - row['Female_Beta']) / se_diff
    p_interaction = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    print(f"{row['Metric']}:")
    print(f"  Sex difference in age effect: {row['Sex_Difference']:.2f} ms/10yr")
    print(f"  P for interaction: {p_interaction:.2e}")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

y_positions = range(len(results_df))
metric_labels = [m.replace('T1_', '').replace('_', ' ') for m in results_df['Metric']]

# Plot females
ax.errorbar(results_df['Female_Beta'], y_positions, 
            xerr=1.96*results_df['Female_SE'],
            fmt='o', color='red', markersize=8, capsize=5, 
            label='Female', linewidth=2, alpha=0.8)

# Plot males
ax.errorbar(results_df['Male_Beta'], 
            [y + 0.2 for y in y_positions],
            xerr=1.96*results_df['Male_SE'],
            fmt='s', color='blue', markersize=8, capsize=5, 
            label='Male', linewidth=2, alpha=0.8)

# Add zero reference line
ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

# Formatting
ax.set_yticks(y_positions)
ax.set_yticklabels(metric_labels, fontsize=11)
ax.set_xlabel('Age Effect (Δ T1 ms per 10 years)', fontsize=13, fontweight='bold')
ax.set_title('Sex-Specific Age Associations Across T1 Distribution Metrics', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=12, frameon=True)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('forest_plot_age_T1_all_metrics.png', dpi=300, bbox_inches='tight')
plt.savefig('forest_plot_age_T1_all_metrics.pdf', bbox_inches='tight')
plt.show()